In [0]:
# Bootstrap

import os
import sys

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
print(notebook_path)

repo_root = "/Workspace" + "/".join(notebook_path.split("/")[:-2])
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print(f"Repo Root: {repo_root}")

In [0]:
# Imports
from pyspark.sql import functions as F
import pyspark.sql.window as Window
from delta.tables import DeltaTable

from src.transformations.gold_transformer import (
    build_fact_transactions,
    build_dim_date,
    build_dim_amount_bucket,
    build_dim_fraud_class,
    build_gold_summary
)

from src.utils.spark_session import get_spark_session, configure_adls_oauth
spark = get_spark_session(app_name="GoldStarSchema")
print("Imports successful")

In [0]:
# Read config
import yaml

config_path = os.path.join(repo_root, "config", "pipeline_config.yml")
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

env = "dev"
cfg = config["environments"][env]

storage_account = cfg["storage_account"]

SILVER_PATH = cfg["silver_path"]
GOLD_PATH = cfg["gold_path"]

PIPELINE_RUN_ID = "gold_manual_run_001"

print("Environment: ", env)
print(f"Silver Path: {SILVER_PATH}")
print(f"Gold Path: {GOLD_PATH}")

In [0]:
# ADLS oauth config

client_id = dbutils.secrets.get(scope="kv-bank-etl-scope", key="databricks-sp-client-id")
client_secret = dbutils.secrets.get(scope="kv-bank-etl-scope", key="databricks-sp-client-secret")
tenant_id = dbutils.secrets.get(scope="kv-bank-etl-scope", key="databricks-sp-tenant-id")

configure_adls_oauth(spark, storage_account, client_id, client_secret, tenant_id)
print("ADLS OAuth config successful")

In [0]:
# Read from silver delta table

df_silver = spark.read.format("delta").load(SILVER_PATH)

print(f"Read {df_silver.count():,} records from silver table")
print(f"Column count is {len(df_silver.columns):,}")
df_silver.printSchema()

In [0]:
# Build fact_transactions

df_fact = df_silver.select(
    "transaction_id",
    F.col("Amount").alias("txn_amount"),
    F.col("Class").alias("fraud_class_key"),
    F.col("Time").alias("transaction_time_seconds"),
    F.col("_ingestion_date").alias("date_key"),
    F.col("amount_bucket").alias("amount_bucket_key"),
    F.col("is_fraud"),
    F.round(F.col("Amount"), 2).alias("amount_rounded"),
    F.col("_ingestion_timestamp"),
    F.col("_source_file"),
    F.col("_silver_processed_timestamp"),
)

print(f"Fact table shape: {(df_fact.count(), len(df_fact.columns))}")

In [0]:
# Build dimensions

df_dim_date = build_dim_date(df_silver)
df_dim_amount = build_dim_amount_bucket(df_silver)
df_dim_fraud = build_dim_fraud_class(spark)

print(f"dim_date rows: {df_dim_date.count():,}")
print(f"dim_amount rows: {df_dim_amount.count():,}")
print(f"dim_fraud rows: {df_dim_fraud.count():,}")

print("\ndim_date sample: ")
df_dim_date.show(5, truncate=False)

print("dim_amount_bucket:")
df_dim_amount.orderBy("bucket_sort_order").show()

print("dim_fraud_class:")
df_dim_fraud.show()

In [0]:
# Write fact_transactions to Gold delta table

FACT_PATH = f"{GOLD_PATH}fact_transactions/"

if DeltaTable.isDeltaTable(spark, FACT_PATH):
    print("Gold Fact Transactions delta table is already present. Running merge logic...")
    
    gold_fact = DeltaTable.forPath(spark, FACT_PATH)

    gold_fact.alias("target").merge(
        df_fact.alias("source"),
        "target.transaction_id = source.transaction_id"
    ).whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
            .execute()

else:
    print("Gold Fact Transactions delta table is not present. Writing new table...")
    df_fact.write \
        .format("delta") \
            .mode("overwrite") \
                .partitionBy("date_key") \
                    .save(FACT_PATH)

print(f"Fact table written to: {FACT_PATH}")

# Verify
df_verify = spark.read.format("delta").load(FACT_PATH)
print(f"Fact Table row count: {df_verify.count():,}")

In [0]:
# Write dimension tables

def write_dim(df, name):
    path = f"{GOLD_PATH}{name}/"
    df.write \
        .format("delta") \
            .mode("overwrite") \
                .save(path)
    count = spark.read.format("delta").load(path).count()
    print(f"{name} written - {count} rows at {path}")

write_dim(df_dim_date, "dim_date")
write_dim(df_dim_amount, "dim_amount_bucket")
write_dim(df_dim_fraud, "dim_fraud_class")

In [0]:
# Build and write gold summary

df_summary = build_gold_summary(df_silver)

SUMMARY_PATH = f"{GOLD_PATH}gold_summary/"

df_summary.write \
    .format("delta") \
        .mode("overwrite") \
            .save(SUMMARY_PATH)
print("Gold Summary written")
df_summary.show(10, truncate=False)

In [0]:
# ── Cell 11: Run analytical queries ──────────────────────

print("=== Gold layer analytics ===\n")

df_fact_q = spark.read.format("delta").load(FACT_PATH)
df_dim_fraud_q = spark.read.format("delta").load(
    f"{GOLD_PATH}dim_fraud_class/"
)
df_dim_amount_q = spark.read.format("delta").load(
    f"{GOLD_PATH}dim_amount_bucket/"
)

# Query 1 — Fraud vs legitimate transaction counts
print("1. Fraud vs legitimate split:")
df_fact_q.groupBy("is_fraud") \
    .agg(
        F.count("transaction_id").alias("count"),
        F.round(F.sum("amount_rounded"), 2).alias("total_amount"),
        F.round(F.avg("amount_rounded"), 2).alias("avg_amount")
    ).show()

# Query 2 — Transaction volume by amount bucket
print("2. Volume by amount bucket:")
df_fact_q.join(
    df_dim_amount_q,
    df_fact_q["amount_bucket_key"] == df_dim_amount_q["amount_bucket_key"]
).groupBy("bucket_name", "bucket_sort_order") \
 .agg(F.count("transaction_id").alias("count")) \
 .orderBy("bucket_sort_order") \
 .show()

# Query 3 — Fraud rate by amount bucket
print("3. Fraud rate by amount bucket:")
df_fact_q.groupBy("amount_bucket_key") \
    .agg(
        F.count("transaction_id").alias("total_count"),
        F.sum(F.col("is_fraud").cast("int")).alias("fraud_count"),
        F.round(
            F.sum(F.col("is_fraud").cast("int")) /
            F.count("transaction_id") * 100, 4
        ).alias("fraud_rate_pct")
    ).orderBy("fraud_rate_pct", ascending=False) \
     .show()